# Planning and Reflection Demo

A small controller that packs pretend orientation kits. Each kit needs one folder and one label.
The stock starts with 3 folders, 2 ready labels and 1 spare label. The goal is 3 packed kits that have been checked.

I compare three things: a fixed plan that fails, a plan I fixed by hand, and a simple rule-based picker.
Everything is made up and runs the same way every time.

In [1]:
print(1 + 1)

2


## Step 1: State and goal

**Prediction:** No, three kits can't be made right away because there are only 2 ready labels. A packed count alone is not enough because the goal also needs `verified` to be True.

In [2]:
from copy import deepcopy

target = 3
initial = {
    "inventory": {"folders": 3, "labels": 2, "spares": 1},
    "packed": 0,
    "verified": False,
}

def goal_met(state, target):
    return state["packed"] == target and state["verified"]

print(initial)
print("Goal met:", goal_met(initial, target))

{'inventory': {'folders': 3, 'labels': 2, 'spares': 1}, 'packed': 0, 'verified': False}
Goal met: False


**What happened:** Goal met is False. One ready label is missing, but there is a spare that could fill the gap. The goal also needs the verified flag, not just the count.

## Step 2: A plan as data

**Prediction:** The plan is missing a restock step. If check ran before assembling anything, it would fail because packed would be 0, not 3.

In [3]:
first_plan = [
    {"action": "inspect", "quantity": 0},
    {"action": "assemble", "quantity": 3},
    {"action": "check", "quantity": 0},
]
for number, proposal in enumerate(first_plan, start=1):
    print(number, proposal)
print("State still unchanged:", initial["packed"] == 0)

1 {'action': 'inspect', 'quantity': 0}
2 {'action': 'assemble', 'quantity': 3}
3 {'action': 'check', 'quantity': 0}
State still unchanged: True


**What happened:** The plan has three steps and nothing in the stock changed. Writing a plan down does not mean it will work.

## Step 3: Inspect and restock tools

**Prediction:** No stock should move. If there aren't enough spares, restock should stop before changing anything.

In [4]:
def inspect_stock(state, quantity, target):
    return "Stock inspected."

def restock(state, quantity, target):
    stock = state["inventory"]
    if stock["spares"] < quantity:
        raise ValueError("not enough spare labels")
    stock["spares"] -= quantity
    stock["labels"] += quantity
    state["verified"] = False
    return f"Moved {quantity} spare label(s)."

demo = deepcopy(initial)
print(restock(demo, 1, target))
print(demo["inventory"])
print("Original labels:", initial["inventory"]["labels"])

Moved 1 spare label(s).
{'folders': 3, 'labels': 3, 'spares': 0}
Original labels: 2


**What happened:** The copy now has folders 3, labels 3, spares 0. The original still has 2 labels, so deepcopy kept them separate.

## Step 4: Assemble and check

**Prediction:** On the original stock, assemble 3 would fail because there are only 2 labels. The goal can't pass before check_kits runs because verified is still False.

In [5]:
def assemble(state, quantity, target):
    stock = state["inventory"]
    if state["packed"] + quantity > target:
        raise ValueError("would exceed the target")
    if stock["folders"] < quantity or stock["labels"] < quantity:
        raise ValueError("not enough ready materials")
    stock["folders"] -= quantity
    stock["labels"] -= quantity
    state["packed"] += quantity
    state["verified"] = False
    return f"Assembled {quantity} kit(s)."

def check_kits(state, quantity, target):
    state["verified"] = state["packed"] == target
    return f"Count check: {state['verified']}"

print(assemble(demo, 3, target))
print("Before check:", goal_met(demo, target))
print(check_kits(demo, 0, target))
print("After check:", goal_met(demo, target))

Assembled 3 kit(s).
Before check: False
Count check: True
After check: True


**What happened:** Three kits were packed in the restocked demo. Before check was False and after check was True. The check only counts kits, it can't tell if real kits are correct.

## Step 5: Validate before running

**Prediction:** `announce_success` will not make the goal true because it isn't a real tool. A quantity of `"3"` will be rejected because it is a string, not an int.

In [6]:
tools = {"inspect": inspect_stock, "restock": restock,
         "assemble": assemble, "check": check_kits}

def execute(proposal, state, target):
    try:
        if type(proposal) is not dict or set(proposal) != {"action", "quantity"}:
            raise ValueError("expected action and quantity")
        name, q = proposal["action"], proposal["quantity"]
        if type(name) is not str or name not in tools:
            raise ValueError("unknown action")
        if type(q) is not int:
            raise ValueError("quantity must be an integer")
        if name in {"inspect", "check"}:
            if q != 0:
                raise ValueError("read/check quantity must be zero")
        elif not 1 <= q <= target:
            raise ValueError("quantity outside task bounds")
        note = tools[name](state, q, target)
        ok = True
    except ValueError as error:
        note, ok = str(error), False
    return {"ok": ok, "note": note, "state": deepcopy(state)}

print(execute({"action": "announce_success", "quantity": 0},
              deepcopy(initial), target)["note"])

unknown action


**What happened:** It printed `unknown action`. Nothing was packed and nothing was verified. Only names in the tools dictionary can run.

## Step 6: Run the first plan

**Prediction:** Two attempts will show up. Inspect works, then assemble fails. Check will never run.

In [7]:
def follow_plan(plan, start, target):
    state = deepcopy(start)
    trace = []
    for proposal in plan:
        observation = execute(proposal, state, target)
        trace.append({"proposal": deepcopy(proposal),
                      "observation": observation})
        if not observation["ok"]:
            break
    return state, trace

failed_state, failed_trace = follow_plan(first_plan, initial, target)
for event in failed_trace:
    print(event["proposal"]["action"], event["observation"]["note"])
print("Packed:", failed_state["packed"])
print("Goal met:", goal_met(failed_state, target))

inspect Stock inspected.
assemble not enough ready materials
Packed: 0
Goal met: False


**What happened:** Inspect worked and assemble failed with `not enough ready materials`. Nothing got packed and the plan stopped. The plan forgot that it needed one more label.

## Step 7: Fix the plan from the evidence

**Prediction:** Adding a restock of 1 label before assembling fixes it. Saying "try harder" would not change any steps, so it would fail the same way.

In [8]:
feedback = {
    "evidence": failed_trace[-1]["observation"]["note"],
    "cause": "The plan tried three kits with two ready labels.",
    "change": "Inspect stock, then restock the one-label shortage.",
}
lessons = [feedback]
revised_plan = [first_plan[0],
                {"action": "restock", "quantity": 1},
                first_plan[1], first_plan[2]]
revised_state, revised_trace = follow_plan(revised_plan, initial, target)
print(feedback["change"])
print([e["proposal"]["action"] for e in revised_trace])
print("Goal met:", goal_met(revised_state, target))

Inspect stock, then restock the one-label shortage.
['inspect', 'restock', 'assemble', 'check']
Goal met: True


**What happened:** The new order was inspect, restock, assemble, check and the goal was met in four attempts. I made this change by hand. The lessons list just stores the feedback, it doesn't do anything by itself.

## Step 8: Pick the next action from what was seen

**Prediction:** It won't restock if there are already enough labels. If the spares can't cover the shortage, it returns None.

In [9]:
def choose_next(observation, target):
    if observation is None:
        return {"action": "inspect", "quantity": 0}
    state = observation["state"]
    remaining = target - state["packed"]
    if remaining == 0:
        return {"action": "check", "quantity": 0}
    if remaining < 0:
        return None
    stock = state["inventory"]
    if stock["folders"] < remaining:
        return None
    shortage = max(0, remaining - stock["labels"])
    if shortage > stock["spares"]:
        return None
    if shortage:
        return {"action": "restock", "quantity": shortage}
    return {"action": "assemble", "quantity": remaining}

observed = execute({"action": "inspect", "quantity": 0},
                   deepcopy(initial), target)
print(choose_next(observed, target))

{'action': 'restock', 'quantity': 1}


**What happened:** It chose restock with quantity 1. This is just if statements, not a language model. It handles this shortage but it doesn't try every possible plan.

## Step 9: A loop with a step limit

**Prediction:** It should be success. The goal is checked right after each action, so success on the 4th action counts before the budget runs out.

In [10]:
def run_agent(start, target, max_steps=4, selector=choose_next):
    if type(target) is not int or not 1 <= target <= 6:
        raise ValueError("target must be an integer from 1 to 6")
    if type(max_steps) is not int or not 1 <= max_steps <= 20:
        raise ValueError("max_steps must be an integer from 1 to 20")
    state, trace = deepcopy(start), []
    observation = None
    if goal_met(state, target):
        return {"status": "success", "state": state, "trace": trace}
    status = "step_limit"
    for _ in range(max_steps):
        proposal = selector(deepcopy(observation), target)
        if proposal is None:
            status = "blocked"
            break
        observation = execute(proposal, state, target)
        trace.append({"proposal": deepcopy(proposal),
                      "observation": observation})
        if not observation["ok"]:
            status = "action_error"
            break
        if goal_met(state, target):
            status = "success"
            break
    return {"status": status, "state": state, "trace": trace}

print("Controller ready.")

Controller ready.


**What happened:** It printed Controller ready. A run can end four ways: success, blocked, action_error or step_limit.

## Step 10: A successful run

**Prediction:** Four attempts: inspect, restock, assemble, check. The proof is the check note saying True and goal_met being True.

In [11]:
result = run_agent(initial, target)
for n, event in enumerate(result["trace"], start=1):
    proposal = event["proposal"]
    observation = event["observation"]
    print(n, proposal["action"], proposal["quantity"],
          observation["note"])
print("Status:", result["status"])
print("Final state:", result["state"])
print("Initial labels:", initial["inventory"]["labels"])

1 inspect 0 Stock inspected.
2 restock 1 Moved 1 spare label(s).
3 assemble 3 Assembled 3 kit(s).
4 check 0 Count check: True
Status: success
Final state: {'inventory': {'folders': 0, 'labels': 0, 'spares': 0}, 'packed': 3, 'verified': True}
Initial labels: 2


**What happened:** It took inspect 0, restock 1, assemble 3, check 0. Packed is 3, verified is True and all stock is 0. The original still has 2 labels. This only proves the simulator run worked.

## Step 11: Stopping rules

**Prediction:** No, packed 3 with verified False is not success. Inspecting over and over never changes anything, so it can never reach the goal.

In [12]:
short_run = run_agent(initial, target, max_steps=3)
no_spares = deepcopy(initial)
no_spares["inventory"]["spares"] = 0
blocked_run = run_agent(no_spares, target)

def repeat_inspect(observation, target):
    return {"action": "inspect", "quantity": 0}

repeat_run = run_agent(initial, target, max_steps=2,
                       selector=repeat_inspect)
print("Short:", short_run["status"], short_run["state"]["packed"],
      short_run["state"]["verified"])
print("No spares:", blocked_run["status"], len(blocked_run["trace"]))
print("Repeated inspection:", repeat_run["status"],
      len(repeat_run["trace"]))

Short: step_limit 3 False
No spares: blocked 1
Repeated inspection: step_limit 2


**What happened:** Short run hit step_limit with 3 packed but not verified. No spares was blocked after one inspect. Repeated inspection hit step_limit after two tries. None of them count as success.

## Step 12: A new case and a fake success claim

**Prediction:** Four kits will take four attempts (inspect, restock, assemble, check). The fake action won't change packed or verified.

In [13]:
new_case = {
    "inventory": {"folders": 4, "labels": 3, "spares": 1},
    "packed": 0, "verified": False,
}
new_result = run_agent(new_case, 4)

def claim_success(observation, target):
    return {"action": "announce_success", "quantity": 0}

claim = run_agent(initial, target, selector=claim_success)
print("New case:", new_result["status"], new_result["state"]["packed"],
      len(new_result["trace"]))
print("Unsupported claim:", claim["status"], claim["state"]["packed"],
      claim["state"]["verified"])

New case: success 4 4
Unsupported claim: action_error 0 False


**What happened:** The new case worked with 4 kits in 4 attempts. The fake claim got action_error with packed 0 and verified False. Just saying "success" can't get past the controller.

## Practice: Two kits

**Prediction:** inspect, assemble 2, check. There are enough labels so no restock is needed. Left over should be 1 folder, 0 labels and 1 spare.

In [14]:
two_run = run_agent(initial, 2, max_steps=3)
for event in two_run["trace"]:
    print(event["proposal"]["action"], event["proposal"]["quantity"],
          event["observation"]["note"])
print("Status:", two_run["status"])
print("Left over:", two_run["state"]["inventory"])

inspect 0 Stock inspected.
assemble 2 Assembled 2 kit(s).
check 0 Count check: True
Status: success
Left over: {'folders': 1, 'labels': 0, 'spares': 1}


**What happened:** It went inspect, assemble 2, check and ended in success. Left over is 1 folder, 0 labels and 1 spare. Success on the third attempt is valid because the goal is checked right after each action, and the check happened before the 3 step budget ran out.

## Practice: Only one folder

**Prediction:** It will inspect, then see there is only 1 folder for 2 kits and stop as blocked.

In [15]:
one_folder = deepcopy(initial)
one_folder["inventory"]["folders"] = 1
folder_run = run_agent(one_folder, 2, max_steps=3)
print("Status:", folder_run["status"], "Attempts:", len(folder_run["trace"]))
print("Last observation:", folder_run["trace"][-1]["observation"]["state"]["inventory"])

Status: blocked Attempts: 1
Last observation: {'folders': 1, 'labels': 2, 'spares': 1}


**What happened:** It was blocked after one inspect. The inspect showed `folders: 1` but 2 kits were still needed. The policy has no tool to get more folders, so it can't go on.

**Grounded feedback:** Evidence: inspect showed 1 folder with 2 kits remaining. Cause: not enough folders and no way to add any. Change: add an "order folders" tool (or ask a person to bring one more folder), or lower the target to 1 kit.

## The four statuses

- **success**: the goal check passed. Packed equals the target and verified is True.
- **blocked**: the selector returned None because its rules have no next step it can take (like not enough folders). It doesn't mean every other strategy would fail.
- **action_error**: an action was rejected, like an unknown action name or a bad quantity. The run stops there.
- **step_limit**: the loop used up all its attempts without reaching the goal. Stopping is not the same as finishing.

## New requirement: correct names on labels

If the labels also had to have the right names, packed and verified would not prove that. They only count kits. The checker would need something like a list of the expected names and a record of the name on each packed label, then compare them one by one. It should be a new goal, not a secret change to the old one, so the old controller isn't graded on a rule it was never built for.

## Report

The first fixed plan failed because it ignored the label shortage. The trace showed `not enough ready materials`, so I added a restock of 1 label and the revised plan worked. The rule-based picker found the same steps on its own from what it saw. These are small made-up cases, so they don't show how well a real language model would plan.